In [1]:
!pip install -q segmentation-models-pytorch
!pip install -q torch-summary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.9 MB/s eta 0:00:0000:01


In [2]:
import os
import random
import time
from pathlib import Path
from tqdm import tqdm
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn
import torchvision.transforms as T
from torchvision.transforms import functional as F

# Проверка CUDA
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def dice_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7, threshold: float = 0.5) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()

def iou_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7, threshold: float = 0.5) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

# =========================
# КОМБИНИРОВАННАЯ ФУНКЦИЯ ПОТЕРЬ
# =========================

class CombinedLoss(nn.Module):
    """
    Комбинированная функция потерь: Dice + BCE + Focal
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.3, focal_weight=0.2):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.bce = nn.BCEWithLogitsLoss()
        self.focal = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
        
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.focal_weight = focal_weight
        
    def forward(self, logits, targets):
        dice_loss = self.dice(logits, targets)
        bce_loss = self.bce(logits, targets)
        focal_loss = self.focal(logits, targets)
        
        total_loss = (self.dice_weight * dice_loss + 
                      self.bce_weight * bce_loss + 
                      self.focal_weight * focal_loss)
        
        return total_loss

# =========================
# АУГМЕНТАЦИИ ЧЕРЕЗ TORCHVISION
# =========================

class TrainTransform:
    """Трансформации для тренировочного датасета"""
    def __init__(self, img_size=512):
        self.img_size = img_size
        
    def __call__(self, image, mask):
        # Конвертируем в тензоры
        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image).float() / 255.0
            mask = torch.from_numpy(mask).float()
        
        # Переставляем размерности для torchvision (C, H, W)
        if image.dim() == 3 and image.shape[-1] == 3:
            image = image.permute(2, 0, 1)
        if mask.dim() == 3 and mask.shape[-1] == 1:
            mask = mask.permute(2, 0, 1)
        elif mask.dim() == 2:
            mask = mask.unsqueeze(0)
        
        # Ресайз
        image = F.resize(image, (self.img_size, self.img_size))
        mask = F.resize(mask, (self.img_size, self.img_size))
        
        # Случайный горизонтальный флип
        if random.random() > 0.5:
            image = F.hflip(image)
            mask = F.hflip(mask)
        
        # Случайный вертикальный флип
        if random.random() > 0.7:
            image = F.vflip(image)
            mask = F.vflip(mask)
        
        # Случайный поворот
        if random.random() > 0.5:
            angle = random.uniform(-30, 30)
            image = F.rotate(image, angle, fill=0)
            mask = F.rotate(mask, angle, fill=0)
        
        # Случайное изменение яркости и контраста
        if random.random() > 0.5:
            brightness = random.uniform(0.8, 1.2)
            contrast = random.uniform(0.8, 1.2)
            image = F.adjust_brightness(image, brightness)
            image = F.adjust_contrast(image, contrast)
        
        # Добавляем случайный шум
        if random.random() > 0.7:
            noise = torch.randn_like(image) * 0.05
            image = image + noise
            image = torch.clamp(image, 0, 1)
        
        # Нормализация для EfficientNet
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        return image, mask

class ValTransform:
    """Трансформации для валидационного датасета (только ресайз и нормализация)"""
    def __init__(self, img_size=512):
        self.img_size = img_size
        
    def __call__(self, image, mask):
        # Конвертируем в тензоры
        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image).float() / 255.0
            mask = torch.from_numpy(mask).float()
        
        # Переставляем размерности
        if image.dim() == 3 and image.shape[-1] == 3:
            image = image.permute(2, 0, 1)
        if mask.dim() == 3 and mask.shape[-1] == 1:
            mask = mask.permute(2, 0, 1)
        elif mask.dim() == 2:
            mask = mask.unsqueeze(0)
        
        # Ресайз
        image = F.resize(image, (self.img_size, self.img_size))
        mask = F.resize(mask, (self.img_size, self.img_size))
        
        # Нормализация
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        return image, mask

# =========================
# DATASET С ПОДДЕРЖКОЙ АУГМЕНТАЦИЙ
# =========================

class BinarySegDataset(Dataset):
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int = 512,
        phase: str = 'train',  # 'train' или 'val'
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.phase = phase
        
        # Выбираем трансформации в зависимости от фазы
        if phase == 'train':
            self.transform = TrainTransform(img_size)
        else:
            self.transform = ValTransform(img_size)
        
        print(f"Loading dataset from {self.images_dir} and {self.masks_dir}")
        print(f"Phase: {phase}, Augmentations: {'ON' if phase == 'train' else 'OFF'}")
        
        self.samples = []
        mask_files = list(self.masks_dir.glob("*.png"))
        print(f"Found {len(mask_files)} mask files")
        
        for mask_path in sorted(mask_files):
            stem = mask_path.stem
            image_path = find_image_for_stem(self.images_dir, stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))
            else:
                print(f"Warning: No image found for mask {stem}")

        if not self.samples:
            raise RuntimeError(f"No paired image/mask samples found")

        print(f"Found {len(self.samples)} paired samples")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]
        
        # Чтение изображения
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise RuntimeError(f"Cannot read image: {image_path}")
        
        # Конвертация BGR -> RGB
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        
        # Чтение маски
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {mask_path}")
        
        # Бинаризация маски
        mask = (mask > 0).astype(np.float32)
        
        # Применяем трансформации
        image, mask = self.transform(image, mask)
        
        return image, mask

# =========================
# МОДЕЛЬ
# =========================

def build_model() -> nn.Module:
    if MODEL_NAME == "Unet":
        model = smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "FPN":
        model = smp.FPN(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")
    return model

# =========================
# ФУНКЦИИ ОБУЧЕНИЯ
# =========================

def train_one_epoch(model, loader, optimizer, loss_fn, device, scaler=None):
    model.train()
    
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    pbar = tqdm(loader, desc="Training", unit="batch")
    
    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = loss_fn(logits, masks)
            
            scaler.scale(loss).backward()
            # Gradient clipping для стабильности
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = loss_fn(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits.detach(), masks, threshold=THRESHOLD)
        running_iou += iou_score_from_logits(logits.detach(), masks, threshold=THRESHOLD)
        
        pbar.set_postfix({
            'loss': f'{running_loss/(batch_idx+1):.4f}',
            'dice': f'{running_dice/(batch_idx+1):.4f}',
        })
    
    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = loss_fn(logits, masks)

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits, masks, threshold=THRESHOLD)
        running_iou += iou_score_from_logits(logits, masks, threshold=THRESHOLD)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

# =========================
# КОНФИГУРАЦИЯ
# =========================

DATA_ROOT = Path("/kaggle/input/competitions/dl-lab-3-product-segmentation/train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2
NUM_WORKERS = 2
SEED = 42
THRESHOLD = 0.5

MODEL_NAME = "UnetPlusPlus"
ENCODER_NAME = "efficientnet-b5"
ENCODER_WEIGHTS = "imagenet"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# ОСНОВНАЯ ФУНКЦИЯ
# =========================

def main():
    seed_everything(SEED)
    
    print("="*50)
    print("Step 1: Creating datasets...")
    print("="*50)
    
    # Создаем тренировочный и валидационный датасеты отдельно
    # Сначала получаем все индексы
    full_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        phase='train'
    )
    
    print("\n" + "="*50)
    print("Step 2: Splitting dataset...")
    print("="*50)
    
    val_size = int(len(full_dataset) * VAL_RATIO)
    train_size = len(full_dataset) - val_size
    
    generator = torch.Generator().manual_seed(SEED)
    train_indices, val_indices = random_split(
        list(range(len(full_dataset))), 
        [train_size, val_size], 
        generator=generator
    )
    
    # Создаем отдельные датасеты с правильными трансформациями
    train_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        phase='train'  # С аугментациями
    )
    
    val_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        phase='val'    # Без аугментаций
    )
    
    # Используем Subset для выбора нужных индексов
    from torch.utils.data import Subset
    train_dataset = Subset(train_dataset, train_indices.indices)
    val_dataset = Subset(val_dataset, val_indices.indices)
    
    print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")
    
    print("\n" + "="*50)
    print("Step 3: Creating data loaders...")
    print("="*50)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE == 'cuda' else False,
        drop_last=False,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE == 'cuda' else False,
        drop_last=False,
    )
    
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    
    # Тестируем загрузку одного батча
    print("\n" + "="*50)
    print("Step 4: Testing data loader...")
    print("="*50)
    try:
        test_batch = next(iter(train_loader))
        print(f"✓ Data loader works!")
        print(f"  Images shape: {test_batch[0].shape}")
        print(f"  Masks shape: {test_batch[1].shape}")
        print(f"  Image range: {test_batch[0].min():.2f} to {test_batch[0].max():.2f}")
        print(f"  Mask unique values: {torch.unique(test_batch[1])}")
    except Exception as e:
        print(f"✗ Data loader failed: {e}")
        return
    
    print("\n" + "="*50)
    print("Step 5: Building model...")
    print("="*50)
    
    model = build_model().to(DEVICE)
    print(f"✓ Model built with {sum(p.numel() for p in model.parameters()):,} parameters")
    print(f"  Model: {MODEL_NAME}")
    print(f"  Encoder: {ENCODER_NAME}")
    
    # Включаем градиентное чекпоинтинг для экономии памяти
    if hasattr(model.encoder, 'set_gradient_checkpointing'):
        model.encoder.set_gradient_checkpointing(True)
        print("✓ Gradient checkpointing enabled")
    
    # Mixed precision training
    scaler = torch.amp.GradScaler('cuda') if DEVICE == "cuda" else None
    if scaler:
        print("✓ AMP (mixed precision) enabled")
    
    # КОМБИНИРОВАННАЯ ФУНКЦИЯ ПОТЕРЬ
    loss_fn = CombinedLoss(dice_weight=0.5, bce_weight=0.3, focal_weight=0.2)
    print("✓ Combined Loss (Dice + BCE + Focal) enabled")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    print("\n" + "="*50)
    print("Step 6: Starting training...")
    print("="*50)
    
    best_val_dice = -1.0
    history = []
    
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch}/{NUM_EPOCHS}")
        print(f"{'='*50}")
        
        epoch_start = time.time()
        
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE, scaler)
        val_metrics = validate_one_epoch(model, val_loader, loss_fn, DEVICE)
        
        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_dice": train_metrics["dice"],
            "train_iou": train_metrics["iou"],
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "val_iou": val_metrics["iou"],
        }
        history.append(row)
        
        epoch_time = time.time() - epoch_start
        print(f"Epoch {epoch} completed in {epoch_time:.1f}s")
        print(f"  Train - Loss: {train_metrics['loss']:.4f}, Dice: {train_metrics['dice']:.4f}, IoU: {train_metrics['iou']:.4f}")
        print(f"  Val   - Loss: {val_metrics['loss']:.4f}, Dice: {val_metrics['dice']:.4f}, IoU: {val_metrics['iou']:.4f}")
        
        # Сохраняем последнюю модель
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": row["val_dice"],
                "config": {
                    "MODEL_NAME": MODEL_NAME,
                    "ENCODER_NAME": ENCODER_NAME,
                    "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                    "IMG_SIZE": IMG_SIZE,
                },
            },
            SAVE_DIR / "last.pth",
        )

        # Сохраняем лучшую модель
        if row["val_dice"] > best_val_dice:
            best_val_dice = row["val_dice"]
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_dice": row["val_dice"],
                    "config": {
                        "MODEL_NAME": MODEL_NAME,
                        "ENCODER_NAME": ENCODER_NAME,
                        "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                        "IMG_SIZE": IMG_SIZE,
                    },
                },
                SAVE_DIR / "best.pth",
            )
            print(f"✓ Saved new best model with val_dice={best_val_dice:.4f}")
        
        # Очищаем кэш GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    print(f"\n{'='*50}")
    print(f"✓ Training finished!")
    print(f"Best val_dice: {best_val_dice:.4f}")
    print(f"Models saved to: {SAVE_DIR}")
    print(f"{'='*50}")

if __name__ == "__main__":
    main()

CUDA available: True
Using GPU: Tesla T4
CUDA version: 12.8
Step 1: Creating datasets...
Loading dataset from /kaggle/input/competitions/dl-lab-3-product-segmentation/train/images and /kaggle/input/competitions/dl-lab-3-product-segmentation/train/masks
Phase: train, Augmentations: ON
Found 2000 mask files
Found 2000 paired samples

Step 2: Splitting dataset...
Loading dataset from /kaggle/input/competitions/dl-lab-3-product-segmentation/train/images and /kaggle/input/competitions/dl-lab-3-product-segmentation/train/masks
Phase: train, Augmentations: ON
Found 2000 mask files
Found 2000 paired samples
Loading dataset from /kaggle/input/competitions/dl-lab-3-product-segmentation/train/images and /kaggle/input/competitions/dl-lab-3-product-segmentation/train/masks
Phase: val, Augmentations: OFF
Found 2000 mask files
Found 2000 paired samples
Train size: 1600, Val size: 400

Step 3: Creating data loaders...
Train batches: 400, Val batches: 100

Step 4: Testing data loader...
✓ Data loader w

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

✓ Model built with 31,910,081 parameters
  Model: UnetPlusPlus
  Encoder: efficientnet-b5
✓ AMP (mixed precision) enabled
✓ Combined Loss (Dice + BCE + Focal) enabled

Step 6: Starting training...

Epoch 1/100


Training: 100%|██████████| 400/400 [02:39<00:00,  2.51batch/s, loss=0.1838, dice=0.7327]


...

Epoch 98 completed in 166.0s
  Train - Loss: 0.0241, Dice: 0.9576, IoU: 0.9222
  Val   - Loss: 0.0647, Dice: 0.9024, IoU: 0.8417

Epoch 99/100


Training: 100%|██████████| 400/400 [02:26<00:00,  2.73batch/s, loss=0.0245, dice=0.9561]


Epoch 99 completed in 165.6s
  Train - Loss: 0.0245, Dice: 0.9561, IoU: 0.9205
  Val   - Loss: 0.0655, Dice: 0.9022, IoU: 0.8414

Epoch 100/100


Training: 100%|██████████| 400/400 [02:26<00:00,  2.73batch/s, loss=0.0243, dice=0.9566]


Epoch 100 completed in 165.8s
  Train - Loss: 0.0243, Dice: 0.9566, IoU: 0.9208
  Val   - Loss: 0.0644, Dice: 0.9027, IoU: 0.8421

✓ Training finished!
Best val_dice: 0.9032
Models saved to: seg_train_runs


In [4]:
import json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

# =========================
# CONFIG
# =========================
TEST_IMAGES_DIR = Path("/kaggle/input/competitions/dl-lab-3-product-segmentation/test_images")
OUTPUT_CSV = "submission.csv"
CHECKPOINT_PATH = Path("./seg_train_runs/best.pth")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# HELPERS
# =========================
def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    return cv2.imread(str(path), flags)

def build_model(model_name: str, encoder_name: str, encoder_weights=None):
    if model_name == "Unet":
        model = smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights, in_channels=3, classes=1, activation=None)
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(encoder_name=encoder_name, encoder_weights=encoder_weights, in_channels=3, classes=1, activation=None)
    elif model_name == "FPN":
        model = smp.FPN(encoder_name=encoder_name, encoder_weights=encoder_weights, in_channels=3, classes=1, activation=None)
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {model_name}")
    return model

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))

def preprocess_image(image: np.ndarray, img_size: int, preprocess_input=None) -> torch.Tensor:
    image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    if preprocess_input is not None:
        image = preprocess_input(image)
    else:
        image = image.astype(np.float32) / 255.0
    image = torch.from_numpy(image.transpose(2, 0, 1)).float()
    return image

@torch.no_grad()
def predict_with_tta(model, image_tensor, device, use_tta=True):
    image_tensor = image_tensor.unsqueeze(0).to(device)
    if not use_tta:
        logits = model(image_tensor)
        return torch.sigmoid(logits)[0, 0]
    
    predictions = []
    # 1. Оригинал
    predictions.append(torch.sigmoid(model(image_tensor))[0, 0])
    # 2. Горизонтальный флип
    predictions.append(torch.flip(torch.sigmoid(model(torch.flip(image_tensor, dims=[-1])))[0, 0], dims=[-1]))
    # 3. Вертикальный флип
    predictions.append(torch.flip(torch.sigmoid(model(torch.flip(image_tensor, dims=[-2])))[0, 0], dims=[-2]))
    # 4. Поворот 90
    predictions.append(torch.rot90(torch.sigmoid(model(torch.rot90(image_tensor, 1, dims=[-2, -1])))[0, 0], -1, dims=[-2, -1]))
    # 5. Поворот 180
    predictions.append(torch.rot90(torch.sigmoid(model(torch.rot90(image_tensor, 2, dims=[-2, -1])))[0, 0], -2, dims=[-2, -1]))
    # 6. Поворот 270
    predictions.append(torch.rot90(torch.sigmoid(model(torch.rot90(image_tensor, 3, dims=[-2, -1])))[0, 0], -3, dims=[-2, -1]))
    
    return torch.stack(predictions).mean(dim=0)

# =========================
# MAIN INFERENCE
# =========================
if __name__ == "__main__":
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT_PATH}")

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    cfg = checkpoint["config"]
    
    model = build_model(cfg["MODEL_NAME"], cfg["ENCODER_NAME"], None)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE).eval()
    
    preprocess_input = get_preprocessing_fn(cfg["ENCODER_NAME"], pretrained=cfg.get("ENCODER_WEIGHTS", "imagenet"))
    image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])

    print(f"Loaded {cfg['MODEL_NAME']}. Processing {len(image_paths)} images...")
    
    rows = []
    for i, img_path in enumerate(image_paths, 1):
        img_bgr = cv2_imread_unicode(img_path)
        if img_bgr is None: continue
        
        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_tensor = preprocess_image(img_rgb, int(cfg["IMG_SIZE"]), preprocess_input)
        
        pred = predict_with_tta(model, img_tensor, DEVICE, use_tta=True).cpu().numpy()
        if pred.shape != (H, W):
            pred = cv2.resize(pred, (W, H), interpolation=cv2.INTER_LINEAR)
        
        mask = (pred > THRESHOLD).astype(np.uint8)
        
        # Очистка шума
        if mask.sum() > 0:
            num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
            for j in range(1, num):
                if stats[j, cv2.CC_STAT_AREA] < 100: mask[labels == j] = 0

        rows.append({"ImageId": img_path.name, "mask": serialize_mask(mask)})
        if i % 100 == 0: print(f"Progress: {i}/{len(image_paths)}")

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    print(f"✓ Done! Saved to {OUTPUT_CSV}")

Loaded UnetPlusPlus. Processing 2000 images...
Progress: 100/2000
Progress: 200/2000
Progress: 300/2000
Progress: 400/2000
Progress: 500/2000
Progress: 600/2000
Progress: 700/2000
Progress: 800/2000
Progress: 900/2000
Progress: 1000/2000
Progress: 1100/2000
Progress: 1200/2000
Progress: 1300/2000
Progress: 1400/2000
Progress: 1500/2000
Progress: 1600/2000
Progress: 1700/2000
Progress: 1800/2000
Progress: 1900/2000
Progress: 2000/2000
✓ Done! Saved to submission.csv
